# SDG 15.3.1 土地退化评估分析

本 notebook 用于计算和评估 SDG 15.3.1 指标（土地退化比例），通过三种不同的指标类型（**trend**、**state**、**performance**）来评估土地生产力变化。

## 评估指标类型

### 1. Trend（趋势）指标

**建模方法：Theil-Sen + Mann-Kendall**

趋势指标用于评估土地生产力的长期变化趋势，使用以下统计方法：

- **Theil-Sen 斜率估计** (`theil_sen_slope_block`)：
  - 计算所有时间点对之间的斜率：`(yj - yi) / (j - i)`
  - 取所有斜率的中位数作为稳健的趋势估计
  - 对异常值具有鲁棒性，是线性趋势的稳健估计

- **Mann-Kendall Z 值** (`mann_kendall_z`)：
  - 计算 Mann-Kendall 检验的 S 统计量：`S = Σ sign(yj - yi)` for all i < j
  - 将 S 转换为标准正态分布的 Z 值
  
  $$S = \sum_{i<j} \text{sign}(y_j - y_i)$$
  
  $$\text{Var}(S) = \frac{T(T-1)(2T+5) - \sum_t t(t-1)(2t+5)}{18}$$
  
  $$Z = \frac{S}{\sqrt{\text{Var}(S)}}$$
  
  - 考虑数据中的并列值（ties）对方差的影响
  - `|Z| > 1.96` 表示在 95% 置信水平下趋势显著

**备选方法：Kendall tau-b 相关系数** (`kendall_tau_b_z`)

Kendall tau-b 相关系数作为备选方法，用于计算时间序列与时间的相关性：

$$\tau_b = \frac{S}{\sqrt{P(P - T_y)}}$$

其中：
- $S = \sum_{i<j} \text{sign}(y_j - y_i)$：Mann-Kendall S 统计量
- $P = \frac{T(T-1)}{2}$：总对数
- $T_y$：y 值中并列值对数的总和

为进行显著性检验，使用 Equation 6 将 $\tau_b$ 转换为 Z 值：

$$Z = \frac{3 \cdot \tau \cdot \sqrt{N(N - 1)}}{2(2N + 5)}$$

其中：
- $\tau$：Kendall Tau-b 相关系数
- $N$：样本数量（时间序列长度 $T$）

**重要说明**：当时间序列在时间域上单调（无并列值）时，Kendall Tau-b 转换后的 Z 值与 Mann-Kendall Z 值在数学上等价。

- **并列值处理**：修正了并列值对相关系数的影响
- **使用方式**：通过 `trend_args.mask = True` 启用此方法（默认使用 Mann-Kendall Z 值）

**分类规则**：
- 根据 Z 值分箱边界将像元分类为退化/改善
- 默认分箱：`[-∞, -1.96, -0.5, 0.5, 1.96, +∞]`
- 最左侧分箱（Z < -1.96）：显著退化
- 最右侧分箱（Z > 1.96）：显著改善

### 2. State（状态）指标

**建模方法：分布一致性检验**

状态指标用于评估当前状态相对于基准期的变化，使用两时期分布一致性检验：

- **分布一致性检验** (`distribution_consistent`)：
  - 将 16 年时间序列分为两个时期：
    - **基准期 H0**：前 13 年（y[:13]）
    - **评估期 H1**：后 3 年（y[13:]）
  - 计算 Z 值：
  
  $$Z = \frac{\bar{y}_1 - \bar{y}_0}{\sigma_0 / \sqrt{n_1}}$$
  
  其中：
  - $\bar{y}_1$：评估期均值
  - $\bar{y}_0$：基准期均值
  - $\sigma_0$：基准期标准差
  - $n_1$：评估期有效样本数
  
  - 此方法检验评估期均值是否显著偏离基准期分布

**生产力差异**：
- 同时计算 `prod_diff = mean₁ - mean₀`
- 正值表示生产力提升，负值表示生产力下降

**分类规则**：
- 根据 Z 值分箱判断退化/改善状态
- 可以应用差异阈值掩膜，过滤变化幅度较小的像元

### 3. Performance（性能）指标

**建模方法：相对性能评估**

性能指标用于评估土地生产力相对于同类生态区域的水平，使用 LCEU（土地覆盖生态单元）分区评估：

- **90 分位数计算** (`calculate_90_quantile`)：
  - 为每个 LCEU 分区计算多年平均 NPP 的 90 分位数
  - LCEU 分区用于定义生态相似的区域
  - 90 分位数作为"高生产力"的参考阈值

- **性能比率计算** (`performance_evaluation`)：
  - 计算每个像元的性能比率：`Performance = mean_npp / P90_lceu`
    - `mean_npp`：该像元的多年平均 NPP
    - `P90_lceu`：该像元所属 LCEU 分区的 90 分位数
  - `Performance > 1.0`：高于分区 90 分位数（高性能）
  - `Performance < 1.0`：低于分区 90 分位数（低性能）

**分类规则**：
- 根据性能比率阈值进行分类（如 `[0, 0.5, 1.0, +∞]`）
- 低于 0.5：显著退化（性能严重不足）
- 高于 1.0：改善（性能优秀）

## 工作流程

1. **数据准备**：初始化参数，设置日志记录
2. **循环处理**：对每个报告年份（2015-2025）执行评估
3. **结果汇总**：生成各年份的退化和改善土地面积统计表格

---

In [3]:
# 导入必要的库和函数
from pathlib import Path
import os
from functions import setup_logger
import numpy as np
import math
import rasterio
from rasterio.windows import Window
from tqdm import tqdm
from functions import mask_low_productivity
import geopandas as gpd
from rasterio.features import geometry_mask
import pandas as pd
from functions import reproject_to_equal_area,classify_bins
from functions import BasicArgs,pixel_area_km2,prepare_profile
from functions import initalize_args
from typing import Any,Mapping,Union
from rasterio.enums import Resampling
from functions import generate_report_period

# 初始化结果列表，用于存储所有年份的评估结果
final_result = []

In [2]:
# 配置日志记录系统
# 日志文件将保存到 logs/trend.log，包含程序运行过程中的详细信息
log_path = Path("logs/trend.log")
os.makedirs(log_path.parent, exist_ok=True)
log = setup_logger(name="sdg", log_file=log_path)

## 1. Trend（趋势）指标评估

本节使用 **Theil-Sen + Mann-Kendall** 方法评估土地生产力的长期变化趋势。

### 方法说明

- **Theil-Sen 斜率**：计算时间序列的趋势斜率，对异常值稳健
- **Mann-Kendall Z 值**：检验趋势的统计显著性（`|Z| > 1.96` 表示显著）

### 处理流程

对每个报告年份（2015-2025）：
1. 初始化评估参数（包括 NPP 数据路径、输出目录等）
2. 调用 `generate_report_period()` 执行完整评估流程：
   - 重投影 NPP 数据到等面积坐标系（EPSG:6933）
   - 计算 Theil-Sen 斜率和 Mann-Kendall Z 值
   - 将结果重投影到等面积坐标系（用于准确面积计算）
   - 根据 Z 值分箱分类退化和改善土地
   - 按酋长国统计退化和改善面积
3. 汇总所有年份的结果到 DataFrame

In [18]:
# Trend 指标评估：计算 2015-2025 年各年份的土地生产力趋势变化
metrics = "trend"
args = None
# 循环处理每个报告年份
for year in range(2015,2026):
    print(f"现在的报告期是{year}")
    # 初始化该年份的评估参数
    args = initalize_args(metrics, year)
    # 生成该年份的评估结果并添加到结果列表
    final_result.append(generate_report_period(metrics, args))

# 将结果列表转换为 DataFrame，便于查看和分析
degrad_by_year = pd.DataFrame(final_result)
outdir = args.out_dir.parent/"overall"
os.makedirs(outdir, exist_ok=True)
degrad_by_year.to_csv(outdir / f"{year}_trend_by_year.csv", index=False)
degrad_by_year


现在的报告期是2015


Emirates: 100%|██████████| 7/7 [00:00<00:00, 77.31it/s]


          Emirate  Degrading land (km²)  Improving land (km²)
0       Abu Dhabi             1360.1250            17368.6250
1           Ajman               10.7500               79.8125
2           Dubai              125.6875             1336.9375
3        Fujairah               87.8750               65.4375
4  Ras Al-Khaimah               50.5625              181.5625
5         Sharjah               84.5000              432.5625
6  Umm al-Qaywayn                5.3750               57.5000

Land Cover Change Statistics
        Land cover change category  Area (km²)  Percent of total land area (%)
Land area with improved land cover     19522.4                            27.5
  Land area with stable land cover     49426.9                            69.5
Land area with degraded land cover      1724.9                             2.4
 Land area with no land cover data       422.7                             0.6
现在的报告期是2016


Emirates: 100%|██████████| 7/7 [00:00<00:00, 102.75it/s]


          Emirate  Degrading land (km²)  Improving land (km²)
0       Abu Dhabi             3632.0625             9847.8750
1           Ajman               25.5625               56.6250
2           Dubai              342.0000              760.6250
3        Fujairah              425.3125               40.2500
4  Ras Al-Khaimah              247.2500               99.6250
5         Sharjah              340.2500              229.1250
6  Umm al-Qaywayn               56.0000               33.1875

Land Cover Change Statistics
        Land cover change category  Area (km²)  Percent of total land area (%)
Land area with improved land cover     11067.3                            15.6
  Land area with stable land cover     54546.1                            76.7
Land area with degraded land cover      5068.4                             7.1
 Land area with no land cover data       415.1                             0.6
现在的报告期是2017


...

Emirates: 100%|██████████| 7/7 [00:00<00:00, 86.13it/s]


          Emirate  Degrading land (km²)  Improving land (km²)
0       Abu Dhabi             4415.6250             4736.6250
1           Ajman               29.8125               53.6250
2           Dubai              255.3125             1001.4375
3        Fujairah               17.2500              654.2500
4  Ras Al-Khaimah               43.6875              555.9375
5         Sharjah               75.7500              504.4375
6  Umm al-Qaywayn               23.3125               41.2500

Land Cover Change Statistics
        Land cover change category  Area (km²)  Percent of total land area (%)
Land area with improved land cover      7547.6                            10.6
  Land area with stable land cover     58277.0                            82.0
Land area with degraded land cover      4860.8                             6.8
 Land area with no land cover data       411.6                             0.6
现在的报告期是2025


Emirates: 100%|██████████| 7/7 [00:00<00:00, 78.11it/s]


          Emirate  Degrading land (km²)  Improving land (km²)
0       Abu Dhabi             2397.9375             5215.4375
1           Ajman               26.4375               47.4375
2           Dubai              229.8125             1086.9375
3        Fujairah               13.1875              682.4375
4  Ras Al-Khaimah               26.3125              663.3750
5         Sharjah               50.9375              609.9375
6  Umm al-Qaywayn               14.2500               60.4375

Land Cover Change Statistics
        Land cover change category  Area (km²)  Percent of total land area (%)
Land area with improved land cover      8366.0                            11.8
  Land area with stable land cover     59565.5                            83.8
Land area with degraded land cover      2758.9                             3.9
 Land area with no land cover data       406.6                             0.6


,Target year,Area of Degrading Land (km²),Area of Improving Land (km²)
0,2015,1724.8750,19522.4375
1,2016,5068.4375,11067.3125
2,2017,4747.4375,11627.3125
3,2018,6301.6250,8767.1875
4,2019,9852.9375,5364.6250
5,2020,10763.6875,3684.1875
6,2021,7849.0000,5174.0625
7,2022,8460.5625,4765.0625
8,2023,6262.8125,6182.6875
9,2024,4860.7500,7547.5625


## 2. State（状态）指标评估

本节使用 **分布一致性检验** 方法评估当前状态相对于基准期的变化。

### 方法说明

- **两时期比较**：
  - 基准期（H0）：前 13 年
  - 评估期（H1）：后 3 年
- **Z 值计算**：`Z = (mean₁ - mean₀) / (std₀ / √n₁)`
  - 检验评估期均值是否显著偏离基准期分布
- **生产力差异**：`prod_diff = mean₁ - mean₀`
  - 直观反映生产力变化幅度

### 处理流程

对每个报告年份（2015-2025）：
1. 初始化评估参数
2. 调用 `generate_report_period()` 执行完整评估流程
3. 汇总结果到 DataFrame

**注意**：State 指标使用 16 年数据窗口（报告年份-15 到报告年份），因此需要确保有足够的历史数据。

In [22]:
# State 指标评估：计算 2015-2025 年各年份的土地生产力状态变化
final_result = []  # 重置结果列表
metrics = "state"

# 循环处理每个报告年份

for year in range(2015,2026):
    print(f"现在的报告期是{year}")
    # 初始化该年份的评估参数
    args = initalize_args(metrics, year)
    # 生成该年份的评估结果并添加到结果列表
    final_result.append(generate_report_period(metrics, args))
# 将结果列表转换为 DataFrame
degrad_by_year = pd.DataFrame(final_result)
outdir = args.out_dir.parent/"overall"
print(outdir)
os.makedirs(outdir, exist_ok=True)
degrad_by_year.to_csv(outdir / f"{year}_state_by_year.csv", index=False)
degrad_by_year
    

现在的报告期是2015


Emirates: 100%|██████████| 7/7 [00:00<00:00, 83.90it/s]


          Emirate  Degrading land (km²)  Improving land (km²)
0       Abu Dhabi              691.8125              910.3125
1           Ajman                4.7500               42.5625
2           Dubai               54.7500              341.6250
3        Fujairah               57.6250               43.0000
4  Ras Al-Khaimah               59.3125               50.8750
5         Sharjah               34.0625              162.2500
6  Umm al-Qaywayn                2.5000               16.8125

Land Cover Change Statistics
        Land cover change category  Area (km²)  Percent of total land area (%)
Land area with improved land cover      1567.4                             2.2
  Land area with stable land cover     68202.0                            95.9
Land area with degraded land cover       904.8                             1.3
 Land area with no land cover data       422.7                             0.6
现在的报告期是2016


Emirates: 100%|██████████| 7/7 [00:00<00:00, 109.65it/s]


          Emirate  Degrading land (km²)  Improving land (km²)
0       Abu Dhabi            10862.6250             4067.0000
1           Ajman               13.6875               72.9375
2           Dubai              483.4375              454.8750
3        Fujairah              298.5625               59.1250
4  Ras Al-Khaimah              126.5625               94.3125
5         Sharjah              187.6875              201.5625
6  Umm al-Qaywayn               21.9375               37.4375

Land Cover Change Statistics
        Land cover change category  Area (km²)  Percent of total land area (%)
Land area with improved land cover      4987.2                             7.0
  Land area with stable land cover     53700.1                            75.5
Land area with degraded land cover     11994.5                            16.9
 Land area with no land cover data       415.1                             0.6
现在的报告期是2017


...

Emirates: 100%|██████████| 7/7 [00:00<00:00, 93.60it/s]


          Emirate  Degrading land (km²)  Improving land (km²)
0       Abu Dhabi             3371.8750             8873.7500
1           Ajman               15.8750              106.1875
2           Dubai              168.7500             2853.0000
3        Fujairah                6.8125             1308.2500
4  Ras Al-Khaimah               14.1250             1428.6250
5         Sharjah               24.6250             1895.7500
6  Umm al-Qaywayn                8.3125              141.0000

Land Cover Change Statistics
        Land cover change category  Area (km²)  Percent of total land area (%)
Land area with improved land cover     16606.6                            23.4
  Land area with stable land cover     50468.4                            71.0
Land area with degraded land cover      3610.4                             5.1
 Land area with no land cover data       411.6                             0.6
现在的报告期是2025


Emirates: 100%|██████████| 7/7 [00:00<00:00, 98.49it/s]


          Emirate  Degrading land (km²)  Improving land (km²)
0       Abu Dhabi             1923.2500            15463.4375
1           Ajman               13.6875               95.8125
2           Dubai              161.6875             3048.6250
3        Fujairah                5.1250             1258.2500
4  Ras Al-Khaimah                9.9375             1485.7500
5         Sharjah               24.8125             1989.8750
6  Umm al-Qaywayn                7.3750              228.2500

Land Cover Change Statistics
        Land cover change category  Area (km²)  Percent of total land area (%)
Land area with improved land cover     23570.0                            33.2
  Land area with stable land cover     44974.5                            63.3
Land area with degraded land cover      2145.9                             3.0
 Land area with no land cover data       406.6                             0.6
../output/state/overall


,Target year,Area of Degrading Land (km²),Area of Improving Land (km²)
0,2015,904.8125,1567.4375
1,2016,11994.5000,4987.2500
2,2017,7327.5000,7116.7500
3,2018,5540.5000,9676.9375
4,2019,5494.3125,7555.5625
5,2020,14642.9375,3703.8125
6,2021,6713.8125,5796.3750
7,2022,4725.4375,8502.3750
8,2023,2634.0625,14535.4375
9,2024,3610.3750,16606.5625


## 3. Performance（性能）指标评估

本节使用 **相对性能评估** 方法，基于 LCEU（土地覆盖生态单元）分区评估土地生产力性能。

### 方法说明

- **LCEU 分区**：将研究区划分为生态相似的区域
- **90 分位数阈值**：为每个 LCEU 分区计算多年平均 NPP 的 90 分位数，作为"高生产力"参考
- **性能比率**：`Performance = mean_npp / P90_lceu`
  - 评估每个像元相对于同类生态区域的生产力水平
  - `> 1.0`：高性能（超过 90% 的同类区域）
  - `< 0.5`：低性能（显著低于平均水平）

### 处理流程

对每个报告年份（2015-2025）：
1. 初始化评估参数（需要读取 LCEU 数据）
2. **设置自定义分箱边界**：`z_edges = [-inf, 0.5, 1.0, inf]`
   - 用于根据性能比率分类退化/改善
3. 调用 `generate_report_period()` 执行完整评估流程
4. 汇总结果到 DataFrame

**注意**：Performance 指标不依赖时间序列趋势，而是基于评估期的平均生产力水平与同类区域的比较。

In [4]:
# Performance 指标评估：计算 2015-2025 年各年份的土地生产力相对性能
final_result = []  # 重置结果列表
metrics = "performance"
args = None

# 循环处理每个报告年份
for year in range(2015,2026):
    print(f"现在的报告期是{year}")
    # 初始化该年份的评估参数
    args = initalize_args(metrics, year)
    # 设置性能比率的分箱边界：
    # - < 0.5: 低性能（退化）
    # - 0.5-1.0: 中等性能
    # - > 1.0: 高性能（改善）
    args.z_edges = [-np.inf, 0.5, 1.0, np.inf]
    # 生成该年份的评估结果并添加到结果列表
    final_result.append(generate_report_period(metrics, args))

# 将结果列表转换为 DataFrame
degrad_by_year = pd.DataFrame(final_result)
outdir = args.out_dir.parent/"Overall"
os.makedirs(outdir, exist_ok=True)
degrad_by_year.to_csv(outdir / f"{year}_performance_by_year.csv", index=False)

现在的报告期是2015


Emirates: 100%|██████████| 7/7 [00:00<00:00, 79.05it/s]


          Emirate  Degrading land (km²)  Improving land (km²)
0       Abu Dhabi              201.6875             5917.0625
1           Ajman                9.0625               27.8750
2           Dubai              102.5000              227.1875
3        Fujairah              133.8125              135.8750
4  Ras Al-Khaimah               99.7500              284.5625
5         Sharjah               41.3750              240.8125
6  Umm al-Qaywayn                1.8750               31.6875

Land Cover Change Statistics
        Land cover change category  Area (km²)  Percent of total land area (%)
Land area with improved land cover      6865.1                             9.7
  Land area with stable land cover     62315.6                            87.6
Land area with degraded land cover       590.1                             0.8
 Land area with no land cover data      1326.2                             1.9
现在的报告期是2016


Emirates: 100%|██████████| 7/7 [00:00<00:00, 88.32it/s]


          Emirate  Degrading land (km²)  Improving land (km²)
0       Abu Dhabi              205.0625             5906.8750
1           Ajman               10.1250               28.8125
2           Dubai              110.2500              232.9375
3        Fujairah              146.5625              135.4375
4  Ras Al-Khaimah              106.2500              278.8750
5         Sharjah               42.1875              241.9375
6  Umm al-Qaywayn                1.8125               31.1875

Land Cover Change Statistics
        Land cover change category  Area (km²)  Percent of total land area (%)
Land area with improved land cover      6856.1                             9.6
  Land area with stable land cover     62295.9                            87.6
Land area with degraded land cover       622.2                             0.9
 Land area with no land cover data      1322.8                             1.9
现在的报告期是2017


...

          Emirate  Degrading land (km²)  Improving land (km²)
0       Abu Dhabi              225.4375             5857.8125
1           Ajman               17.9375               33.1250
2           Dubai              193.6250              281.8125
3        Fujairah              210.6875              142.1250
4  Ras Al-Khaimah              155.3750              232.5000
5         Sharjah               54.0625              243.3125
6  Umm al-Qaywayn                2.9375               28.9375

Land Cover Change Statistics
        Land cover change category  Area (km²)  Percent of total land area (%)
Land area with improved land cover      6819.6                             9.6
  Land area with stable land cover     62093.1                            87.3
Land area with degraded land cover       860.1                             1.2
 Land area with no land cover data      1324.2                             1.9
现在的报告期是2024


Emirates: 100%|██████████| 7/7 [00:00<00:00, 95.59it/s]


          Emirate  Degrading land (km²)  Improving land (km²)
0       Abu Dhabi               230.500             5855.1875
1           Ajman                18.375               33.5625
2           Dubai               203.250              291.3750
3        Fujairah               213.000              142.1250
4  Ras Al-Khaimah               155.375              226.6875
5         Sharjah                55.500              245.8125
6  Umm al-Qaywayn                 2.875               28.9375

Land Cover Change Statistics
        Land cover change category  Area (km²)  Percent of total land area (%)
Land area with improved land cover      6823.7                             9.6
  Land area with stable land cover     62071.2                            87.3
Land area with degraded land cover       878.9                             1.2
 Land area with no land cover data      1323.1                             1.9
现在的报告期是2025


Emirates: 100%|██████████| 7/7 [00:00<00:00, 93.21it/s]


          Emirate  Degrading land (km²)  Improving land (km²)
0       Abu Dhabi              232.9375             5861.2500
1           Ajman               18.0000               33.0000
2           Dubai              189.8750              299.0625
3        Fujairah              206.0625              139.1250
4  Ras Al-Khaimah              132.6250              217.5000
5         Sharjah               50.6875              249.7500
6  Umm al-Qaywayn                2.5000               29.5000

Land Cover Change Statistics
        Land cover change category  Area (km²)  Percent of total land area (%)
Land area with improved land cover      6829.2                             9.6
  Land area with stable land cover     62111.0                            87.4
Land area with degraded land cover       832.7                             1.2
 Land area with no land cover data      1324.1                             1.9


各国可以选择使用其偏好的任何指标组合及其重要性评估方法来确定其国界内的土地退化程度。然而，这些选择应在随附的报告中加以说明，最好附上地图，比较使用不同组合方法识别出的退化程度。报告期内识别出的土地生产力退化程度应与基线期或上一个报告期内绘制的、在当前评估中仍然存在的退化程度相加

## 4. 组合三个指标的退化判断

本节根据 **Table 4-4** 或 **Table 4-5** 的查找表规则，组合 trend、state、performance 三个指标的分类结果，判断每个像素点是否退化。

### 方法说明

**查找表规则（Table 4-4 和 Table 4-5）**：

根据三个指标（Trend、State、Performance）的退化状态（Y=退化，N=非退化），判断最终是否退化：

- **Class 1**: Trend(Y), State(Y), Performance(Y) → Degraded(Y)
- **Class 2**: Trend(Y), State(Y), Performance(N) → Degraded(Y)
- **Class 3**: Trend(Y), State(N), Performance(Y) → Degraded(Y)
- **Class 4**: Trend(Y), State(N), Performance(N) → Table 4-4: Degraded(Y), Table 4-5: Degraded(N) ⚠️
- **Class 5**: Trend(N), State(Y), Performance(Y) → Degraded(Y)
- **Class 6**: Trend(N), State(Y), Performance(N) → Degraded(N)
- **Class 7**: Trend(N), State(N), Performance(Y) → Table 4-4: Degraded(N), Table 4-5: Degraded(Y) ⚠️
- **Class 8**: Trend(N), State(N), Performance(N) → Degraded(N)

**关键差异**：
- **Class 4**：Table 4-4 判定为退化，Table 4-5 判定为非退化
- **Class 7**：Table 4-4 判定为非退化，Table 4-5 判定为退化

### 处理流程

对每个报告年份（2015-2025）：
1. 直接使用已保存的 degrading.tif 文件（从 `generate_report_tables` 自动生成）
2. 根据查找表规则判断每个像素是否最终退化
3. 输出 boolean 结果文件（1=退化，0=非退化）

**注意**：
- 只对三个指标都有有效值的像素进行比较（排除无效值）
- 输入文件格式：uint8，1=退化，0=非退化或无效值
- 输出文件格式：uint8，1=最终判定为退化，0=非退化或无效值

In [6]:
# 组合三个指标的退化判断
from functions import combine_metrics_degradation

# 选择一个报告年份进行示例
reporting_year = 2023

# 定义三个指标的输出目录和degrading文件路径（这些文件在 generate_report_tables 中已自动保存）
trend_degrading_path = Path(f"../output/trend/{reporting_year}/degrading.tif")
state_degrading_path = Path(f"../output/state/{reporting_year}/degrading.tif")
performance_degrading_path = Path(f"../output/performance/{reporting_year}/degrading.tif")

# 检查文件是否存在
if not all([trend_degrading_path.exists(), state_degrading_path.exists(), performance_degrading_path.exists()]):
    print(f"警告：部分degrading文件不存在，请先运行前面的trend、state和performance评估！")
    print(f"Trend degrading: {trend_degrading_path.exists()}")
    print(f"State degrading: {state_degrading_path.exists()}")
    print(f"Performance degrading: {performance_degrading_path.exists()}")
else:
    print(f"✓ 所有degrading文件已存在")
    print(f"  Trend degrading: {trend_degrading_path}")
    print(f"  State degrading: {state_degrading_path}")
    print(f"  Performance degrading: {performance_degrading_path}")

✓ 所有degrading文件已存在
  Trend degrading: ../output/trend/2023/degrading.tif
  State degrading: ../output/state/2023/degrading.tif
  Performance degrading: ../output/performance/2023/degrading.tif


In [8]:
# 使用 Table 4-4 标准组合三个指标
output_table44 = Path(f"../output/combined_degradation_{reporting_year}_table44.tif")
result_path_table44 = combine_metrics_degradation(
    trend_degrading_path=trend_degrading_path,
    state_degrading_path=state_degrading_path,
    performance_degrading_path=performance_degrading_path,
    output_path=output_table44,
    table_version="4-4",
)

print(f"Table 4-4 结果已保存至: {result_path_table44}")

# 使用 Table 4-5 标准组合三个指标
output_table45 = Path(f"../output/combined_degradation_{reporting_year}_table45.tif")
result_path_table45 = combine_metrics_degradation(
    trend_degrading_path=trend_degrading_path,
    state_degrading_path=state_degrading_path,
    performance_degrading_path=performance_degrading_path,
    output_path=output_table45,
    table_version="4-5",
)

print(f"Table 4-5 结果已保存至: {result_path_table45}")

# 读取并显示统计信息
from functions import pixel_area_km2
with rasterio.open(result_path_table44) as src:
    degradation_table44 = src.read(1).astype(bool)
    px_area_km2 = pixel_area_km2(src)
    total_degraded_km2_table44 = degradation_table44.sum() * px_area_km2
    
with rasterio.open(result_path_table45) as src:
    degradation_table45 = src.read(1).astype(bool)
    total_degraded_km2_table45 = degradation_table45.sum() * px_area_km2

print(f"\n报告年份: {reporting_year}")
print(f"Table 4-4 标准 - 退化土地面积: {total_degraded_km2_table44:.2f} km²")
print(f"Table 4-5 标准 - 退化土地面积: {total_degraded_km2_table45:.2f} km²")
print(f"差异: {abs(total_degraded_km2_table44 - total_degraded_km2_table45):.2f} km²")

Table 4-4 结果已保存至: ../output/combined_degradation_2023_table44.tif
Table 4-5 结果已保存至: ../output/combined_degradation_2023_table45.tif

报告年份: 2023
Table 4-4 标准 - 退化土地面积: 189053.81 km²
Table 4-5 标准 - 退化土地面积: 189053.81 km²
差异: 0.00 km²


In [7]:
# 批量处理所有年份（2015-2025）
for year in range(2015, 2026):
    print(f"\n处理报告年份: {year}")
    
    # 定义degrading文件路径
    trend_degrading_path = Path(f"../output/trend/{year}/degrading.tif")
    state_degrading_path = Path(f"../output/state/{year}/degrading.tif")
    performance_degrading_path = Path(f"../output/performance/{year}/degrading.tif")
    
    # 检查输入文件是否存在
    if not all([trend_degrading_path.exists(), state_degrading_path.exists(), performance_degrading_path.exists()]):
        print(f"  跳过 {year}：缺少degrading文件")
        print(f"    Trend: {trend_degrading_path.exists()}, State: {state_degrading_path.exists()}, Performance: {performance_degrading_path.exists()}")
        continue
    
    # 使用 Table 4-4 标准
    output_table44 = Path(f"../output/overall/combined_degradation_{year}_table44.tif")
    combine_metrics_degradation(
        trend_degrading_path=trend_degrading_path,
        state_degrading_path=state_degrading_path,
        performance_degrading_path=performance_degrading_path,
        output_path=output_table44,
        table_version="4-4",
    )
    print(f"  Table 4-4 结果已保存")
    
    # 使用 Table 4-5 标准
    output_table45 = Path(f"../output/overall/combined_degradation_{year}_table45.tif")
    combine_metrics_degradation(
        trend_degrading_path=trend_degrading_path,
        state_degrading_path=state_degrading_path,
        performance_degrading_path=performance_degrading_path,
        output_path=output_table45,
        table_version="4-5",
    )
    print(f"  Table 4-5 结果已保存")

print("\n所有年份处理完成！")


处理报告年份: 2015
  Table 4-4 结果已保存
  Table 4-5 结果已保存

处理报告年份: 2016
  Table 4-4 结果已保存
  Table 4-5 结果已保存

处理报告年份: 2017
  Table 4-4 结果已保存
  Table 4-5 结果已保存

处理报告年份: 2018
  Table 4-4 结果已保存
  Table 4-5 结果已保存

处理报告年份: 2019
  Table 4-4 结果已保存
  Table 4-5 结果已保存

处理报告年份: 2020
  Table 4-4 结果已保存
  Table 4-5 结果已保存

处理报告年份: 2021
  Table 4-4 结果已保存
  Table 4-5 结果已保存

处理报告年份: 2022
  Table 4-4 结果已保存
  Table 4-5 结果已保存

处理报告年份: 2023
  Table 4-4 结果已保存
  Table 4-5 结果已保存

处理报告年份: 2024
  Table 4-4 结果已保存
  Table 4-5 结果已保存

处理报告年份: 2025
  Table 4-4 结果已保存
  Table 4-5 结果已保存

所有年份处理完成！


In [20]:
# 读本地 shp
import geemap
gdf = gpd.read_file(REGIONS_ID)

# 强烈建议：先转成 EPSG:4326（EE最稳）
gdf = gdf.to_crs("EPSG:4326")

# 转成 EE FeatureCollection（不用上传）
regions = geemap.geopandas_to_ee(gdf)